### SQLite

É uma biblioteca de software que fornece um sistema de gerenciamento de banco de dados relacional (SGBD).

Principais Características:
- Autocontido: Não precisa de instalação ou configuração complexa. Basta apontar para o arquivo e começar a usar. Enquanto outros funcionam no modelo cliente-servidor (onde você precisa instalar um programa servidor e se conectar a ele), o SQLite é serverless.
- Portabilidade: Como o banco é um único arquivo, você pode movê-lo entre Windows, Mac, Linux ou Android apenas copiando o arquivo. 
- Confiabilidade (ACID): Mesmo sendo leve, ele garante que as transações sejam seguras. Se o computador travar ou a energia acabar, os dados não são corrompidos.
- Domínio Público: Ele é gratuito para qualquer uso, inclusive comercial.

Quando não utilizar:
- Alta Concorrência de Escrita: O SQLite utiliza um sistema de bloqueio no arquivo. **Evite se:** Seu site ou app terá centenas de usuários salvando informações ao mesmo tempo. 
- Grandes Volumes de Dados (Terabytes): Embora o SQLite suporte tecnicamente arquivos de até 140 TB, gerenciar um único arquivo desse tamanho é um risco operacional. **Evite se:** Você precisa distribuir os dados em vários discos ou servidores (sharding). Sistemas como o PostgreSQL lidam muito melhor com volumes massivos e oferecem ferramentas de backup em tempo real mais robustas.
- Acesso Remoto via Rede: O SQLite foi feito para acesso local: **Evite se:** O banco de dados precisa estar em um servidor "A" e a aplicação em um servidor "B".
- Controle de Acesso e Permissões (Usuários): Diferente de outros SGBDs, o SQLite não possui o conceito de "usuários" ou "permissões" dentro do banco. No Sqlite quem tem acesso ao arquivo, tem acesso a tudo **Evite se:** A sua aplicação requer gerenciamento de permissão baseado no usuário, ex: Usuário X só pode ler, Usuário Y pode apagar.
- Recursos Avançados de SQL: Não possui recursos avançados como Stored Procedures, funções trigonométricas nativas avançadas ou tipos de dados extremamente específicos.

In [1]:
import sqlite3
from sqlite3 import Error

In [2]:
# Criar a conexão com o banco de dados, se não existir o arquivo será criado
conn = sqlite3.connect('aula02.db')

In [3]:
# definindo um iterador que permitirá navegar e manipular os registros
cur = conn.cursor()

Criando tabelas no banco de dados

*books*: Tabela que armazenará livros, contará com as seguintes colunas:

- id: chave primária única que identificará um livro
- title: título do livro
- author_id: id do autor do livro (chave estrangeira)

*authors*: Tabela que armazenerá autores, contará com as seguintes colunas

- id: chave primária única que identificará o autor
- name: nome do autor
- born_date: data de nascimento do autor

In [4]:
sql_create_authors_table = """
CREATE TABLE IF NOT EXISTS authors(
	id integer PRIMARY KEY,
	name text NOT NULL,
	born_date text
);
"""

sql_create_books_table = """
CREATE TABLE IF NOT EXISTS books(
	id integer PRIMARY KEY,
	title text NOT NULL,
	author_id integer NOT NULL,
	FOREIGN KEY (author_id) REFERENCES authors (id)
);
"""

In [5]:
def create_table(conn, create_table_sql):
    try:
        c = conn.cursor()
        c.execute(create_table_sql)
    except Error as e:
        print(e)

In [6]:
create_table(conn, sql_create_authors_table)
create_table(conn, sql_create_books_table)

Criando registros no banco de dados

In [7]:
def create_records(conn, table, values):
    if table == "authors":
        sql = "INSERT INTO authors(name, born_date) VALUES (?,?)"
    elif table == "books":
        sql = "INSERT INTO books(title, author_id) VALUES (?,?)"
    if len(values)==2:
        with conn:cur.execute(sql, values)
    else:
        with conn:cur.executemany(sql, values)
    return cur.lastrowid

In [8]:
with conn:
    author_values = ('Aldous Huxley', '1894-07-26')
    author_id = create_records(conn, "authors", author_values)

    book_values = [
		('Brave New World', author_id),
		('The Perennial Philosophy', author_id),
		('The Doors of Perception', author_id),
		('The Art of Seeing', author_id),
		('Update Field', 2)
    ]

    create_records(conn, "books", book_values)

In [10]:
cur.execute('SELECT * from books')
result = cur.fetchall()
print(result)

[(1, 'Brave New World', 1), (2, 'The Perennial Philosophy', 1), (3, 'The Doors of Perception', 1), (4, 'The Art of Seeing', 1), (5, 'Update Field', 2)]


In [11]:
for row in result:
    print(row)

(1, 'Brave New World', 1)
(2, 'The Perennial Philosophy', 1)
(3, 'The Doors of Perception', 1)
(4, 'The Art of Seeing', 1)
(5, 'Update Field', 2)
